# Pemrosesan Data: Pembersihan dan Pembagian Data

## Latar Belakang

Tahap ini melaksanakan langkah-langkah pemrosesan yang telah ditetapkan pada eksplorasi data sebelumnya. Data mentah hasil pengumpulan masih memuat berbagai noise dan artikel berkualitas rendah, sehingga perlu dibersihkan dan disaring sebelum dapat digunakan untuk pemodelan. Selain itu, data juga perlu dibagi menjadi subset latih, validasi, dan uji sebagai persiapan untuk pelatihan model.

## Tujuan

Tahap ini bertujuan mengubah 2.000 artikel mentah (`data/raw/articles.csv`) menjadi dataset bersih (`data/processed/articles_clean.csv`), sekaligus menghasilkan pembagian data stratified 80/10/10 (`data/splits/{train,val,test}.csv`) yang akan digunakan pada pelatihan model klasifikasi. Pemrosesan pada tahap ini terbatas pada pembersihan dan pembagian data; pelabelan sentimen dilakukan pada notebook terpisah dengan model yang dilatih sendiri pada dataset SmSA, dan label kategori dipertahankan sesuai sumbernya (`inet`, bukan teknologi).

In [1]:
!pip install -q pandas scikit-learn matplotlib

In [2]:
import re
from pathlib import Path
import pandas as pd

def find_project_root():
    cwd = Path.cwd().resolve()
    if (cwd / "data").exists() and (cwd / "notebooks").exists():
        return cwd
    for parent in cwd.parents:
        if (parent / "data").exists() and (parent / "notebooks").exists():
            return parent
    return cwd

PROJECT_ROOT = find_project_root()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_SPLITS = PROJECT_ROOT / "data" / "splits"
for d in [DATA_PROCESSED, DATA_SPLITS]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input       : {DATA_RAW / 'articles.csv'}")

Project root: C:\Users\user\news-rag-project
Input       : C:\Users\user\news-rag-project\data\raw\articles.csv


Tahap ini menyiapkan kebutuhan dasar sebelum pemrosesan dijalankan, yaitu pustaka `re` untuk operasi berbasis pola dan `pandas` untuk pengolahan data tabular. Lokasi root proyek dideteksi secara otomatis agar jalur berkas tetap konsisten, dan folder keluaran untuk data terproses (`data/processed`) serta pembagian data (`data/splits`) disiapkan. Data mentah `articles.csv` ditetapkan sebagai masukan untuk tahap ini.

## Load Data Mentah

In [3]:
df = pd.read_csv(DATA_RAW / "articles.csv")
print(f"Total artikel mentah: {len(df)}")
print(f"\nDistribusi awal:")
print(df["kategori"].value_counts())

Total artikel mentah: 2000

Distribusi awal:
kategori
news       400
finance    400
sport      400
inet       400
health     400
Name: count, dtype: int64


Data mentah dimuat dari `articles.csv`, berisi 2.000 artikel dengan distribusi yang seimbang sebanyak 400 artikel pada masing-masing dari lima kategori. Distribusi awal ini dicatat sebagai acuan untuk memastikan keseimbangan tetap terjaga setelah proses pembersihan dan penyaringan dilakukan.

## Cleaning Noise

In [4]:
def clean_text(text):
    """Hapus noise residu Detik."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\[Gambas[^\]]*\]", " ", text)
    text = re.sub(r"\[Gambar[^\]]*\]", " ", text)
    text = re.sub(r"(Lihat|Simak|Tonton)\s+juga\s+video\s*['\"][^'\"]*['\"]\s*:?",
                  " ", text, flags=re.IGNORECASE)
    text = re.sub(r"(Lihat|Simak|Tonton)\s+juga\s+video", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"Baca selengkapnya di sini\s*\.?", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"Baca juga\s*:?[^.]*\.?", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df_clean = df.copy()
df_clean["isi"] = df_clean["isi"].apply(clean_text)
df_clean["judul"] = df_clean["judul"].apply(clean_text)

print("Verifikasi sisa noise:")
for name, pat in [("Gambas", r"\[Gambas"), ("Lihat juga video", r"Lihat juga video"),
                  ("Simak juga video", r"Simak juga video"), ("Tonton juga video", r"Tonton juga video"),
                  ("Baca juga", r"Baca juga")]:
    n = df_clean["isi"].str.contains(pat, case=False, na=False, regex=True).sum()
    print(f"  {name:<20}: {n}")
print("\nCleaning selesai.")

Verifikasi sisa noise:
  Gambas              : 0
  Lihat juga video    : 0
  Simak juga video    : 0
  Tonton juga video   : 0
  Baca juga           : 0

Cleaning selesai.


Berdasarkan pola noise yang teridentifikasi pada eksplorasi data, fungsi `clean_text` menghapus sisa elemen non-konten dari teks menggunakan ekspresi reguler, yaitu penanda video tersemat `[Gambas:...]` dan `[Gambar:...]`, ajakan "Lihat/Simak/Tonton juga video", serta frasa "Baca juga" dan "Baca selengkapnya". Setelah pola-pola tersebut dibuang, spasi berlebih dirapikan agar teks menjadi konsisten. Pembersihan diterapkan pada kolom isi maupun judul. Verifikasi setelah pembersihan menunjukkan seluruh pola noise telah berhasil dihilangkan dari data.


## Penyaringan Artikel Pendek

In [5]:
df_clean["n_words"] = df_clean["isi"].str.split().str.len()
before = len(df_clean)
df_clean = df_clean[df_clean["n_words"] >= 50].copy().reset_index(drop=True)
print(f"Drop < 50 kata: {before - len(df_clean)} dropped -> {len(df_clean)} sisa")

Drop < 50 kata: 139 dropped -> 1861 sisa


Jumlah kata pada setiap artikel dihitung ulang berdasarkan isi yang telah dibersihkan, kemudian artikel dengan panjang di bawah 50 kata disaring. Sebanyak 139 artikel dibuang sehingga tersisa 1.861 artikel. Sesuai temuan eksplorasi data, artikel sependek ini umumnya berupa galeri foto atau konten placeholder yang tidak memuat isi berita yang memadai, sehingga penghapusannya meningkatkan kualitas korpus.

## Drop Duplikat (keep-first)

In [6]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["isi"]).reset_index(drop=True)
print(f"Drop dup isi  : {before - len(df_clean)} -> {len(df_clean)}")
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["judul"]).reset_index(drop=True)
print(f"Drop dup judul: {before - len(df_clean)} -> {len(df_clean)}")
print(f"\n=== Distribusi final ===")
print(df_clean["kategori"].value_counts())

Drop dup isi  : 20 -> 1841
Drop dup judul: 1 -> 1840

=== Distribusi final ===
kategori
news       397
finance    375
sport      366
inet       359
health     343
Name: count, dtype: int64


Duplikat dihapus berdasarkan isi terlebih dahulu, lalu berdasarkan judul, dengan mempertahankan kemunculan pertama. Tahap ini membuang 20 artikel berisi sama dan 1 artikel berjudul sama, sehingga dataset final berjumlah 1.840 artikel. Jumlah duplikat yang lebih kecil dibandingkan temuan eksplorasi data sebelumnya disebabkan sebagian duplikat (terutama galeri foto placeholder) telah lebih dulu tersaring pada tahap penghapusan artikel pendek.

Distribusi akhir antarkategori tetap relatif seimbang, berkisar dari 343 artikel (health) hingga 397 artikel (news). Pergeseran kecil dari distribusi awal yang merata merupakan konsekuensi wajar dari pembersihan, dan keseimbangan yang terjaga ini tetap memadai untuk pelatihan model klasifikasi kategori.

In [7]:
out_path = DATA_PROCESSED / "articles_clean.csv"
df_clean.to_csv(out_path, index=False)
print(f"Saved -> {out_path} ({df_clean.shape})")
print(f"\n--- Panjang artikel (kata) ---")
print(df_clean.groupby("kategori")["n_words"].describe()[["count","mean","min","max"]].round(0).to_string())

Saved -> C:\Users\user\news-rag-project\data\processed\articles_clean.csv ((1840, 7))

--- Panjang artikel (kata) ---
          count   mean    min     max
kategori                             
finance   375.0  360.0  105.0  1715.0
health    343.0  386.0   51.0  1070.0
inet      359.0  400.0   73.0  2051.0
news      397.0  368.0   62.0  1573.0
sport     366.0  265.0   62.0   708.0


Dataset bersih disimpan ke `articles_clean.csv` dengan total 1.840 artikel. Statistik panjang artikel setelah pembersihan menunjukkan nilai minimum yang kini berada di atas ambang 50 kata pada seluruh kategori, menandakan artikel sangat pendek telah berhasil tersaring. Rata-rata panjang artikel berkisar antara 265 kata (sport) hingga 400 kata (inet), dengan sport tetap menjadi kategori berartikel terpendek dan inet cenderung terpanjang, sejalan dengan pola yang teramati pada eksplorasi data.

## Stratified Split (buat Day 3 Fine-tuning)

In [8]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df_clean, test_size=0.2, stratify=df_clean["kategori"], random_state=42)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["kategori"], random_state=42)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"\n--- Distribusi kategori per split ---")
print(pd.DataFrame({
    "train": train_df["kategori"].value_counts(),
    "val": val_df["kategori"].value_counts(),
    "test": test_df["kategori"].value_counts()}))

train_df.to_csv(DATA_SPLITS / "train.csv", index=False)
val_df.to_csv(DATA_SPLITS / "val.csv", index=False)
test_df.to_csv(DATA_SPLITS / "test.csv", index=False)
print(f"\nSaved splits ke {DATA_SPLITS}")

Train: 1472 | Val: 184 | Test: 184

--- Distribusi kategori per split ---
          train  val  test
kategori                  
finance     300   37    38
health      274   35    34
inet        287   36    36
news        318   39    40
sport       293   37    36

Saved splits ke C:\Users\user\news-rag-project\data\splits


Dataset bersih dibagi menjadi tiga subset dengan proporsi 80/10/10 untuk pelatihan, validasi, dan pengujian, menghasilkan 1.472 artikel latih, 184 validasi, dan 184 uji. Pembagian dilakukan secara stratified berdasarkan kategori (dengan `random_state` tetap demi reproduktibilitas), sehingga proporsi setiap kategori terjaga konsisten di seluruh subset sebagaimana terlihat pada tabel distribusi. Stratifikasi ini penting agar model dilatih dan dievaluasi pada komposisi kelas yang representatif. Ketiga subset disimpan ke folder `data/splits/` sebagai masukan untuk pelatihan model klasifikasi pada tahap berikutnya.

Tahap pemrosesan data telah menghasilkan dataset bersih berisi 1.840 artikel (`articles_clean.csv`) beserta pembagian stratified 80/10/10 (`data/splits/`). Noise telah dihapus, artikel pendek dan duplikat telah disaring, dan keseimbangan antarkategori tetap terjaga. Data ini siap digunakan pada tahap berikutnya, yaitu pelatihan model klasifikasi kategori yang menjadi komponen routing pada sistem RAG.